# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print general info
print(f"Name: {dataset.metadata.name if hasattr(dataset.metadata, 'name') else ''}")
print(f"Description: {dataset.metadata.description if hasattr(dataset.metadata, 'description') else ''}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSets and their @id and fields
print("Available RecordSets (by @id):")
record_sets = dataset.record_sets
for recordset in record_sets:
    print(f"  RecordSet @id: {recordset.id}")
    print(f"    Name: {recordset.name}")
    print(f"    Fields (with @id and type):")
    for field in recordset.fields:
        print(f"      Field @id: {field.id} ({field.data_type}) | Label: {field.name}")

# Pick the first record set for downstream exploration
if len(record_sets) > 0:
    primary_record_set_id = record_sets[0].id
    print(f"\nWe will use the first record set: {primary_record_set_id}")
else:
    raise ValueError("No record sets found in the dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from available record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  - {len(df)} records loaded. Columns: {df.columns.tolist()}")

# For demonstration, we use the first available record set
print(f"\nColumns in the main DataFrame for {primary_record_set_id}:")
print(dataframes[primary_record_set_id].columns.tolist())
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for EDA
main_df = dataframes[primary_record_set_id]
numeric_candidates = []
for field in dataset.record_set(primary_record_set_id).fields:
    if field.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number', 'schema:Number']:
        numeric_candidates.append(field.id)
print(f"Numeric fields in RecordSet {primary_record_set_id} by @id: {numeric_candidates}")

if len(numeric_candidates) == 0:
    raise ValueError("No numeric fields found for EDA.")

# For demonstration, use the first numeric field
numeric_field_id = numeric_candidates[0]

# Try to convert column to numeric (might need this if loaded as string)
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notna().sum() > 0 else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records from '{primary_record_set_id}' where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric column
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized field `{numeric_field_id}` for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by first available non-numeric field, if possible
group_candidates = [f.id for f in dataset.record_set(primary_record_set_id).fields if f.data_type not in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number', 'schema:Number']]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"\nGrouping by field: {group_field_id}")
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    group_field_id = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the (filtered, not-NaN) main numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field_id].dropna().hist(bins=30)
plt.title(f'Histogram of {numeric_field_id} (filtered)')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id exists, boxplot by group
if group_field_id and group_field_id in filtered_df.columns:
    import seaborn as sns
    plt.figure(figsize=(10,4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to fetch and analyze the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from stimulated human mast cells using the `mlcroissant` library:

- We inspected metadata, record sets, and fields programmatically using their `@id` fields.
- We extracted all records and loaded each record set into Pandas DataFrames.
- Exploratory Data Analysis (EDA) was performed on a selected numeric field, with filtering, normalization, and optional grouping by a categorical field.
- Basic data visualizations (histograms and, if available, boxplots) illustrated the numeric field distribution and relationships by group.

You can adapt this notebook for downstream proteomics or machine learning workflows, leveraging the `mlcroissant` library to access fields by `@id` for reproducible, schema-driven scientific analysis.